# locket smoke test

File-based inter-process locks. NOTE: `locket`'s `acquire()` returns `None` (not a bool),
so never gate `release()` on its return value; doing so leaves the lock held and a waiting
thread blocked. Locks on the same path within one process share one underlying lock object
and are NOT reentrant.

In [ ]:
# Example 1: context-manager locking — acquire, hold, auto-release on exit.
import locket, os, tempfile

work_path = tempfile.mkdtemp()
lock_path = os.path.join(work_path, "resource.lock")

with locket.lock_file(lock_path):
    assert os.path.exists(lock_path), "lock file must exist while held"
    print("lock held; critical section runs")

# Re-acquiring non-blockingly proves the exit released it.
with locket.lock_file(lock_path, timeout=0):
    print("OK — lock was released on context exit (non-blocking reacquire succeeded)")

In [ ]:
# Example 2: non-blocking acquisition — a held lock fails fast with LockError.
# locket locks are per-path within a process and NOT reentrant; acquire() returns None.
import locket, os, tempfile

lock_path = os.path.join(tempfile.mkdtemp(), "nonblock.lock")

holder = locket.lock_file(lock_path)
holder.acquire()  # returns None — do NOT gate release() on it
try:
    try:
        with locket.lock_file(lock_path, timeout=0):
            raise AssertionError("timeout=0 acquire must fail while the lock is held")
    except locket.LockError:
        print("OK — LockError raised immediately for a held lock (timeout=0)")
finally:
    holder.release()

with locket.lock_file(lock_path, timeout=0):
    print("OK — acquirable again after release")

In [ ]:
# Example 3: timeout semantics under thread contention (bounded, no infinite waits).
import locket, os, tempfile, threading, time

lock_path = os.path.join(tempfile.mkdtemp(), "timeout.lock")
holding = threading.Event()

def hold_briefly():
    with locket.lock_file(lock_path):
        holding.set()
        time.sleep(0.6)

t = threading.Thread(target=hold_briefly)
t.start()
assert holding.wait(timeout=10), "background thread never acquired the lock"

try:
    with locket.lock_file(lock_path, timeout=0.05):
        raise AssertionError("short timeout must expire while the thread holds the lock")
except locket.LockError:
    print("OK — 50 ms timeout expired while the lock was held")

start = time.monotonic()
with locket.lock_file(lock_path, timeout=10):
    print(f"OK — acquired after the holder released ({time.monotonic() - start:.2f}s wait)")
t.join(timeout=10)
assert not t.is_alive(), "holder thread must have finished"

In [ ]:
# Example 4: cross-process exclusion — the package's raison d'etre (fcntl file lock).
import locket, os, subprocess, sys, tempfile, time

lock_path = os.path.join(tempfile.mkdtemp(), "xproc.lock")
marker = lock_path + ".held"
child_code = (
    "import locket, pathlib, time\n"
    f"with locket.lock_file({lock_path!r}):\n"
    f"    pathlib.Path({marker!r}).touch()\n"
    "    time.sleep(1.5)\n"
)
child = subprocess.Popen([sys.executable, "-c", child_code])
deadline = time.monotonic() + 20
while not os.path.exists(marker) and time.monotonic() < deadline:
    time.sleep(0.05)
assert os.path.exists(marker), "child process never acquired the lock"

try:
    with locket.lock_file(lock_path, timeout=0.1):
        raise AssertionError("parent must NOT get the lock while the child holds it")
except locket.LockError:
    print("OK — cross-process exclusion enforced (LockError in parent)")

assert child.wait(timeout=30) == 0, "child process failed"
with locket.lock_file(lock_path, timeout=10):
    print("OK — parent acquired the lock after the child exited")

In [ ]:
# Example 5: exception safety — the context manager releases the lock on error.
import locket, os, tempfile

lock_path = os.path.join(tempfile.mkdtemp(), "err.lock")

try:
    with locket.lock_file(lock_path):
        raise ValueError("simulated failure inside the critical section")
except ValueError:
    print("exception propagated out of the critical section")

# If the lock leaked, this would raise LockError.
with locket.lock_file(lock_path, timeout=0):
    print("OK — lock was released despite the exception")